#  Environment setup  

In [ ]:
!pip install --no-deps bitsandbytes accelerate xformers peft trl==0.22.2 triton

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 161.2 MB/s eta 0:00:00


In [ ]:
!pip install --no-deps cut_cross_entropy unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.1 MB/s eta 0:00:00


In [ ]:
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.7.4 requires msgspec, which is not installed.
unsloth-zoo 2026.7.4 requires tyro; sys_platform != "darwin" or platform_machine != "arm64", which is not installed.
unsloth-zoo 2026.7.4 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.0.0 which is incompatible.
unsloth-zoo 2026.7.4 requires torchao>=0.13.0; sys_platform != "darwin" or platform_machine != "arm64", but you have torchao 0.10.0 which is incompatible.
unsloth-zoo 2026.7.4 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.13.1 which is incompatible.


In [ ]:
!pip install "transformers==4.56.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 134.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 69.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.7.4 requires msgspec, which is not installed.
unsloth-zoo 2026.7.4 requires tyro; sys_platform != "darwin" or platform_machine != "arm64", which is not installed.
unsloth-zoo 2026.7.4 requires data

In [ ]:
!pip install --no-deps unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 MB 7.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import unsloth
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import gc
import json
import time
from pathlib import Path

In [ ]:
import torch
from PIL import Image
from trl import SFTConfig, SFTTrainer

In [ ]:
DRIVE_ARCHIVE = Path("/content/drive/MyDrive/aegis_art_atelier/archives/Aegis-Art-Atelier-22K.tar.gz")
DRIVE_SPLITS_DIR = Path("/content/drive/MyDrive/aegis_art_atelier/splits")

In [ ]:
LOCAL_ROOT = Path("/content")
LOCAL_DATASET_DIR = LOCAL_ROOT / "Aegis-Art-Atelier-22K"
SANITY_CHECK_SAMPLES_PER_SPLIT = 5
SANITY_CHECK_SEED = 42

# Extract the dataset archive
!mkdir -p "{LOCAL_ROOT}"
!tar -xzf "{DRIVE_ARCHIVE}" -C "{LOCAL_ROOT}"

print(f"Extracted data to: {LOCAL_DATASET_DIR}")

Extracted data to: /content/Aegis-Art-Atelier-22K


In [ ]:
archive_size_gb = DRIVE_ARCHIVE.stat().st_size / (1024 ** 3)
print(f"\nArchivo a extraer: {DRIVE_ARCHIVE.name} ({archive_size_gb:.2f} GB)")
print("Se necesita aprox. el doble de espacio libre (archivo comprimido + extraido).")


Archivo a extraer: Aegis-Art-Atelier-22K.tar.gz (3.80 GB)
Se necesita aprox. el doble de espacio libre (archivo comprimido + extraido).


In [ ]:
assert LOCAL_DATASET_DIR.exists(), (
    f"La extraccion no genero la carpeta esperada: {LOCAL_DATASET_DIR}. "
    "Revisa el nombre del directorio dentro del tar.gz."
)
print("OK.")

OK.


In [ ]:
import shutil

In [ ]:
for split_name in ("train", "val", "test"):
    source = DRIVE_SPLITS_DIR / f"{split_name}_manifest.jsonl"
    destination = LOCAL_DATASET_DIR / f"{split_name}_manifest.jsonl"
    shutil.copy2(source, destination)
    print(f"Copiado: {source.name} -> {destination}")

Copiado: train_manifest.jsonl -> /content/Aegis-Art-Atelier-22K/train_manifest.jsonl
Copiado: val_manifest.jsonl -> /content/Aegis-Art-Atelier-22K/val_manifest.jsonl
Copiado: test_manifest.jsonl -> /content/Aegis-Art-Atelier-22K/test_manifest.jsonl


In [ ]:
LOCAL_DATASET_DIR = Path("/content/Aegis-Art-Atelier-22K")
TRAIN_MANIFEST = LOCAL_DATASET_DIR / "train_manifest.jsonl"

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-unsloth-bnb-4bit"

In [ ]:
BENCH_SUBSET_SIZE = 400   # small slice of train, enough for a stable it/s reading
BENCH_STEPS = 30          # short run: pure throughput measurement, no save, no eval

In [ ]:
INSTRUCTION_TEXT = "Describe this artwork in detail: subject, style, composition, and mood."

In [ ]:
# %% Cell 3 - Load a small subset of the training manifest for benchmarking
def load_manifest(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


In [ ]:
train_records = load_manifest(TRAIN_MANIFEST)
print(f"Full train manifest: {len(train_records)} records")

Full train manifest: 18919 records


In [ ]:
bench_records = train_records[:BENCH_SUBSET_SIZE]
print(f"Benchmark subset: {len(bench_records)} records")


Benchmark subset: 400 records


In [ ]:
# %%   Convert manifest records into Unsloth's conversation format %%
# Images are opened once and kept in memory for the benchmark subset only.
# This is not how the full training run will load data (that should stream
# from disk to avoid holding 19k images in RAM); it is fine here because the
# subset is small and this cell exists only to measure throughput.

In [ ]:
def convert_to_conversation(record: dict, base_dir: Path) -> dict:
    image_path = base_dir / record["image_path"]
    image = Image.open(image_path).convert("RGB")

    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": INSTRUCTION_TEXT},
                    {"type": "image", "image": image},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": record["caption_clean"]}],
            },
        ]
    }


In [ ]:
bench_dataset = [
    convert_to_conversation(record, LOCAL_DATASET_DIR) for record in bench_records
]
print(f"Converted {len(bench_dataset)} examples to conversation format")



Converted 400 examples to conversation format


In [ ]:
# %%   Benchmark function: fresh model load, no save, no eval %%
# A fresh model is loaded for every configuration tested, per Root Cause 4
# in the post-mortem: resuming/reusing a warmed-up state changes the VRAM
# baseline and would make the comparison between configs unreliable.

In [ ]:
def run_benchmark(per_device_batch_size: int, gradient_accumulation_steps: int, steps: int) -> dict:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model, tokenizer = FastVisionModel.from_pretrained(
        MODEL_NAME,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
        max_seq_length=2048,
    )

    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        random_state=3407,
    )

    FastVisionModel.for_training(model)

    training_args = SFTConfig(
        output_dir="/content/bench_tmp",
        per_device_train_batch_size=per_device_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        max_steps=steps,
        save_strategy="no",
        eval_strategy="no",
        logging_steps=steps,
        warmup_steps=0,
        learning_rate=2e-4,
        bf16=True,
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=bench_dataset,
        args=training_args,
    )

    start = time.monotonic()
    trainer.train()
    elapsed_seconds = time.monotonic() - start

    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    effective_batch = per_device_batch_size * gradient_accumulation_steps
    steps_per_second = steps / elapsed_seconds

    result = {
        "per_device_batch_size": per_device_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "effective_batch_size": effective_batch,
        "steps": steps,
        "elapsed_seconds": round(elapsed_seconds, 2),
        "steps_per_second": round(steps_per_second, 4),
        "peak_vram_gb": round(peak_vram_gb, 2),
    }

    del trainer
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return result

In [ ]:
CANDIDATE_CONFIGS = [
    {"per_device_batch_size": 8,  "gradient_accumulation_steps": 2},
    {"per_device_batch_size": 16, "gradient_accumulation_steps": 1},
    {"per_device_batch_size": 32, "gradient_accumulation_steps": 1},
]

In [ ]:
# %% C  Run the benchmark across candidate batch configurations
# Effective batch size is kept close to 16 across all three configs, so the
# comparison isolates the effect of per-device batch size vs. VRAM headroom,


In [ ]:
benchmark_results = []

In [ ]:
for config in CANDIDATE_CONFIGS:
    print(f"\nBenchmarking: per_device_batch_size={config['per_device_batch_size']}, "
          f"gradient_accumulation_steps={config['gradient_accumulation_steps']}")
    result = run_benchmark(
        per_device_batch_size=config["per_device_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        steps=BENCH_STEPS,
    )
    print(result)
    benchmark_results.append(result)


Benchmarking: per_device_batch_size=8, gradient_accumulation_steps=2
==((====))==  Unsloth 2026.7.4: Fast Qwen2_5_Vl patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 8,332,536,832 (0.48% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
30,1.732200


{'per_device_batch_size': 8, 'gradient_accumulation_steps': 2, 'effective_batch_size': 16, 'steps': 30, 'elapsed_seconds': 102.9, 'steps_per_second': 0.2916, 'peak_vram_gb': 14.28}

Benchmarking: per_device_batch_size=16, gradient_accumulation_steps=1
==((====))==  Unsloth 2026.7.4: Fast Qwen2_5_Vl patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Model does not have a default image size - using 512


...

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
{'per_device_batch_size': 16, 'gradient_accumulation_steps': 1, 'effective_batch_size': 16, 'steps': 30, 'elapsed_seconds': 110.71, 'steps_per_second': 0.271, 'peak_vram_gb': 16.16}

Benchmarking: per_device_batch_size=32, gradient_accumulation_steps=1
==((====))==  Unsloth 2026.7.4: Fast Qwen2_5_Vl patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 40,370,176 of 8,332,536,832 (0.48% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
30,1.740200


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
{'per_device_batch_size': 32, 'gradient_accumulation_steps': 1, 'effective_batch_size': 32, 'steps': 30, 'elapsed_seconds': 216.65, 'steps_per_second': 0.1385, 'peak_vram_gb': 22.28}


In [ ]:
# %%   Select the best configuration  %%
# Per Root Cause 3: prefer the highest steps_per_second among configs that
# keep peak VRAM comfortably below the ceiling where Unsloth starts
# offloading gradients (roughly 40 percent of 95 GB, about 38 GB).

In [ ]:
VRAM_CEILING_GB = 90

In [ ]:
safe_results = [r for r in benchmark_results if r["peak_vram_gb"] < VRAM_CEILING_GB]
candidates = safe_results if safe_results else benchmark_results

best_result = max(candidates, key=lambda r: r["steps_per_second"])


In [ ]:

print("\nAll benchmark results:")
for r in benchmark_results:
    flag = "SAFE" if r["peak_vram_gb"] < VRAM_CEILING_GB else "NEAR CEILING"
    print(f"  batch={r['per_device_batch_size']} x accum={r['gradient_accumulation_steps']} "
          f"| {r['steps_per_second']} steps/s | {r['peak_vram_gb']} GB VRAM | {flag}")

print(f"\nSelected configuration: per_device_batch_size="
      f"{best_result['per_device_batch_size']}, gradient_accumulation_steps="
      f"{best_result['gradient_accumulation_steps']}")
print(f"Measured throughput: {best_result['steps_per_second']} steps/s")
print(f"Peak VRAM: {best_result['peak_vram_gb']} GB")



All benchmark results:
  batch=8 x accum=2 | 0.2916 steps/s | 14.28 GB VRAM | SAFE
  batch=16 x accum=1 | 0.271 steps/s | 16.16 GB VRAM | SAFE
  batch=32 x accum=1 | 0.1385 steps/s | 22.28 GB VRAM | SAFE

Selected configuration: per_device_batch_size=8, gradient_accumulation_steps=2
Measured throughput: 0.2916 steps/s
Peak VRAM: 14.28 GB


In [ ]:
# %  Project the credit budget for the full training run
NUM_TRAIN_EXAMPLES = len(train_records)
EPOCHS = 2
CREDITS_PER_HOUR = 9
# Blackwell NVIDIA hardware

In [ ]:
effective_batch_size = best_result["effective_batch_size"]
steps_per_epoch = NUM_TRAIN_EXAMPLES // effective_batch_size
total_steps = steps_per_epoch * EPOCHS

In [ ]:
estimated_seconds = total_steps / best_result["steps_per_second"]
estimated_hours = estimated_seconds / 3600
estimated_credits = estimated_hours * CREDITS_PER_HOUR

In [ ]:

print(f"\nTrain examples: {NUM_TRAIN_EXAMPLES}")
print(f"Effective batch size: {effective_batch_size}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total steps for {EPOCHS} epochs: {total_steps}")
print(f"Estimated wall clock: {estimated_hours:.2f} hours")
print(f"Estimated credits (compute only, excludes eval/save overhead): {estimated_credits:.1f}")
print("\nNote: this projection does not yet include eval and checkpoint overhead.")
print("Add eval_steps and save_steps timing once the full training config is set.")



Train examples: 18919
Effective batch size: 16
Steps per epoch: 1182
Total steps for 2 epochs: 2364
Estimated wall clock: 2.25 hours
Estimated credits (compute only, excludes eval/save overhead): 20.3

Note: this projection does not yet include eval and checkpoint overhead.
Add eval_steps and save_steps timing once the full training config is set.
